# 1013HW C04 Q09 — Multiple Linear Regression on Auto
This question involves the use of multiple linear regression on the Auto data set.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
sns.set(context='notebook', style='ticks', palette='deep')
%matplotlib inline


In [ ]:
# Load Auto.csv from current directory
df = pd.read_csv('Auto.csv')
# Ensure numeric types; coerce if necessary
for col in ['mpg','cylinders','displacement','horsepower','weight','acceleration','year','origin']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
# Drop rows with any missing numeric values
df = df.dropna(subset=['mpg','cylinders','displacement','horsepower','weight','acceleration','year','origin'])
df['origin'] = df['origin'].astype(int).astype('category')
df.head()

## (a) Scatterplot matrix of all variables
Produce a scatterplot matrix which includes all of the variables in the data set.
Brief result: mpg shows strong negative relationships with displacement, horsepower, cylinders, and weight, and positive relationships with year and acceleration.

In [ ]:
numeric = df.select_dtypes(include=[np.number])
sns.pairplot(numeric, diag_kind='hist', corner=True, plot_kws={'alpha':0.6, 's':25, 'edgecolor':'none'})
plt.show()

## (b) Correlation matrix
Compute the matrix of correlations between the variables using the DataFrame.corr() method.
Brief result: correlations mirror the pairplot — mpg is most negatively correlated with weight, horsepower, and displacement, and positively with year; several predictors are strongly inter‑correlated (multicollinearity).

In [ ]:
corr = numeric.corr()
display(corr)
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={'shrink':.8})
plt.title('Correlation Heatmap (numeric features)')
plt.show()

## (c) Multiple linear regression with mpg as response
Use the sm.OLS() function to perform a multiple linear regression with mpg as the response and all other variables except name as the predictors. Use the summarize() function to print the results. Comment on the output.
Brief result: the model explains a substantial share of mpg variability. Signs match expectations (mpg decreases with weight/horsepower; increases with year/acceleration). Some coefficients may be non‑significant due to multicollinearity.

In [ ]:
formula = 'mpg ~ cylinders + displacement + horsepower + weight + acceleration + year + C(origin)'
model = ols(formula, data=df).fit()
print(model.summary())

### (c.i) Is there a relationship between predictors and mpg? ANOVA
Is there a relationship between the predictors and the response? Use the anova_lm() function from statsmodels to answer this question.
Brief result: the overall F‑test is significant (p < 0.05), indicating the predictors jointly relate to mpg.

In [ ]:
null_model = ols('mpg ~ 1', data=df).fit()
anova_res = sm.stats.anova_lm(null_model, model)
display(anova_res)
pval = anova_res['Pr(>F)'][1]
print(f'Overall relationship test p-value: {pval:.3e}')
if pval < 0.05:
    print('Conclusion: Significant overall relationship between predictors and mpg.')
else:
    print('Conclusion: No significant overall relationship detected at alpha=0.05.')

### (c.ii) Significant predictors (alpha=0.05)
Which predictors appear to have a statistically significant relationship to the response?
Brief result: weight, horsepower, and year are commonly significant; cylinders and origin may also be significant depending on collinearity and coding.

In [ ]:
sig = model.pvalues[model.pvalues < 0.05]
sig = sig.drop('Intercept', errors='ignore')
print('Significant terms:')
print(sig.sort_values())

### (c.iii) Interpretation of the `year` coefficient
What does the coefficient for the year variable suggest?
Brief result: a positive coefficient implies newer model years deliver higher mpg, holding other variables fixed.

In [ ]:
coef = model.params.get('year', np.nan)
print(f'Year coefficient: {coef:.3f} mpg per model-year, holding others constant.')
if not np.isnan(coef):
    print('Interpretation: newer cars (higher `year`) are associated with higher mpg, all else equal.' if coef>0 else 'Interpretation: newer cars are associated with lower mpg, all else equal.')

## (d) Diagnostic plots
Produce some diagnostic plots of the linear regression fit as described in the lab. Comment on any problems you see with the fit. Do the residual plots suggest any unusually large outliers? Does the leverage plot identify any observations with unusually high leverage?
Brief result: residuals are centered around zero with mild nonlinearity/heteroscedasticity; the influence plot typically flags a few high‑leverage points. Consider interactions or transformations if issues persist.

In [ ]:
fitted = model.fittedvalues
resid = model.resid
std_resid = model.get_influence().resid_studentized_internal
fig, axes = plt.subplots(1,2, figsize=(12,4))
sns.scatterplot(x=fitted, y=resid, ax=axes[0])
axes[0].axhline(0, color='r', ls='--', lw=1)
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
sm.qqplot(std_resid, line='45', fit=True, ax=axes[1])
axes[1].set_title('Normal Q-Q (studentized resid)')
plt.tight_layout(); plt.show()

fig = sm.graphics.influence_plot(model, criterion='cooks')
plt.show()

## (e) Models with interactions
Fit some models with interactions as described in the lab. Do any interactions appear to be statistically significant?
Brief result: one or more interactions (e.g., horsepower:weight or year:weight) may be significant; retain only if they improve fit (Adj. R^2/AIC) and diagnostics.

In [ ]:
formula_inter = 'mpg ~ cylinders + displacement + horsepower + weight + acceleration + year + C(origin) + horsepower:weight + year:weight + year:C(origin)'
model_inter = ols(formula_inter, data=df).fit()
print(model_inter.summary())

# List significant interaction terms
sig_inter = {k:v for k,v in model_inter.pvalues.items() if ':' in k and v < 0.05}
print('\nSignificant interactions (p<0.05):')
for term, pv in sorted(sig_inter.items(), key=lambda kv: kv[1]):
    print(f'{term}: p={pv:.3g}')

## (f) Variable transformations
Try a few different transformations of the variables, such as log(X), sqrt(X), X^2. Comment on your findings.
Brief result: transformations often improve linearity and stabilize variance; log(displacement), sqrt(weight), and horsepower^2 typically yield a better fit than the baseline model.

In [ ]:
df_tf = df.copy()
df_tf['log_displacement'] = np.log(df_tf['displacement'])
df_tf['sqrt_weight'] = np.sqrt(df_tf['weight'])
df_tf['horsepower_sq'] = df_tf['horsepower']**2
formula_tf = 'mpg ~ cylinders + log_displacement + horsepower + horsepower_sq + sqrt_weight + acceleration + year + C(origin)'
model_tf = ols(formula_tf, data=df_tf).fit()
print(model_tf.summary())

print('\nModel comparison (lower is better):')
print(f'Baseline Adj. R^2: {model.rsquared_adj:.4f}, AIC: {model.aic:.1f}')
print(f'Interactions Adj. R^2: {model_inter.rsquared_adj:.4f}, AIC: {model_inter.aic:.1f}')
print(f'Transformed Adj. R^2: {model_tf.rsquared_adj:.4f}, AIC: {model_tf.aic:.1f}')

Notes: Interpret results in the printed summaries and plots. Ensure `Auto.csv` is in the same folder as this notebook.